In [50]:
from FormUtils import pyForm, capture_physics_expr

## One-loop QED vacuum polarization (electron bubble)

We compute the one-loop correction to the photon propagator coming from an electron loop. 
The object of interest is the tensor integral

## Dirac trace

After evaluating the trace, the integrand becomes

$ \Pi^{\mu\nu}(q) \sim \int \frac{d^d k}{(2\pi)^d}
\frac{
-4 k^\mu (k-q)^\nu
-4 k^\nu (k-q)^\mu
-4 g^{\mu\nu} m^2
+4 g^{\mu\nu} k\cdot (k-q)
}{
(k^2 - m^2)((k-q)^2 - m^2)
} $

---

## Propagators and rewriting

We define propagators:

- $ D_1 = k^2 - m^2 $
- $ D_2 = (k - q)^2 - m^2 $

and rewrite all scalar products in terms of them:

$ k \cdot q = \frac{D_1 - D_2 + q^2}{2} $

This step rewrites the numerator entirely in terms of:
- $k^\mu k^\nu$
- $k^\mu$
- scalar quantities

so that standard integral reductions can be applied.

---

## Ward identity

We check gauge invariance by contracting with $q_\mu$:

$ q_\mu \Pi^{\mu\nu}(q) = 0 $

In the code this is `WardInt`, and the result

$ \text{WardInt} = 0 $

confirms the calculation is consistent.

---

## Tensor integral reduction

We reduce all loop integrals to standard scalar functions.

### Rank-2 tensor integral

$ \int \frac{k^\mu k^\nu}{D_1 D_2} = g^{\mu\nu} B_{00}(q^2,m,m) + q^\mu q^\nu B_{11}(q^2,m,m) $

### Rank-1 vector integral

$ \int \frac{k^\mu}{D_1 D_2}
= q^\mu B_1(q^2,m,m) $

### Scalar integral
$ \int \frac{1}{D_1 D_2}
= B_0(q^2,m,m) $

---

##Tadpoles

Single-propagator integrals give tadpoles:

### Scalar tadpole
$ \int \frac{1}{D_1} = A_0(m), \quad
\int \frac{1}{D_2} = A_0(m) $

### Vector tadpole
- Symmetric case:
  $ \int \frac{k^\mu}{D_1} = 0 $

- Shifted case:
To evaluate the rank-1 integral over $D_2$, we perform the shift $k \to k + q$:

$\int \frac{d^d k}{(2\pi)^d} \frac{k^\mu}{(k-q)^2 - m^2} = \int \frac{d^d k}{(2\pi)^d} \frac{(k+q)^\mu}{k^2 - m^2}= \underbrace{\int \frac{d^d k}{(2\pi)^d} \frac{k^\mu}{k^2 - m^2}}_{0 \text{ (Symmetry)}} + q^\mu \underbrace{\int \frac{d^d k}{(2\pi)^d} \frac{1}{k^2 - m^2}}_{A_0(m)}$

Thus, or the shifted propagator is:
$\int \frac{k^\mu}{D_2} = q^\mu A_0(m)$


---

## — Final result

After all replacements, the loop becomes

$ \Pi^{\mu\nu}(q) = g^{\mu\nu} \left( 4 A_0(m) - 2 q^2 B_0(q^2,m,m) - 8 B_{00}(q^2,m,m) \right) + q^\mu q^\nu \left( 8 B_1(q^2,m,m) - 8 B_{11}(q^2,m,m) \right) $

---

In [51]:
%%pyForm ExperimentalLoop

* Process: ExperimentalLoop

#-
* Above suppresses extra output
Off Statistics;
Off FinalStats;
Symbol d;
Dimension d;
#include FeynHelpers.h

Vectors k1,k2;
Symbols e, Mmuon, Melec;
Symbols y, D1, D2shift, bubbleProp;
Symbols qmu1, qmu2, gmu1mu2;
CF  A, A0, B, B0, B00, B1, B11;


Local Bubble = (-e^2) 
             * g(i1, i2, mu1) * fprop(i2, i3, k1, Melec) 
             * g(i3, i4, mu2) * fprop(i4, i1, k2, Melec);
#call Propagators()
#call doTrace(1,0)
.sort;
id k2 = k1 - q;
Bracket prop;
Print +s Bubble ;
.sort

* Replace propagators with D1, D2shift 
id prop( - Melec^2 + k1.k1) = 1/D1 ;
id prop( - Melec^2 + k2.k2) = 1/D2shift;
.sort
Local Numerator = Bubble * D1 * D2shift  ;
#message ">>>"
Print +s Numerator ;

.sort
* Drop the bubble for now
Drop Bubble;
.sort;

* shift/replace loop momentum
id k1.k1 = D1 + Melec^2;
id k1.q  = (D1 - D2shift + q.q)/2;
.sort
#message ">>>"
Print +s Numerator ;
.sort;

Local WardNum = q(mu1) * Numerator;
.sort
* Re-apply the scalar product identity to the new q.k1 terms
id k1.q = (D1 - D2shift + q.q)/2;
.sort
#message ">>>"
Print +s Numerator ;
Print +s WardNum ;
.sort;

* Put the propagators back to turn the numerator into an integral
Local WardInt= WardNum * D1^-1 * D2shift^-1;
Local BubbleInt=  Numerator * D1^-1 * D2shift^-1;
.sort
Drop Numerator, WardNum;
.sort
* Apply the cancellations
id D1 * D1^-1 * D2shift^-1 = D2shift^-1;
id D2shift * D1^-1 * D2shift^-1 = D1^-1;
.sort
#message ">>>"
Print +s WardInt ;
Print +s BubbleInt ;
.sort;

* Now do Integral replacement

* Tensor Integral (Rank 2)
id k1(mu1?)*k1(mu2?)*D1^-1*D2shift^-1 = d_(mu1,mu2)*B00(q.q,Melec,Melec) 
                                 + q(mu1)*q(mu2)*B11(q.q,Melec,Melec);
.sort
* Vector Integral (Rank 1)
id k1(mu1?)*D1^-1*D2shift^-1 = q(mu1)*B1(q.q,Melec,Melec);
.sort
* Scalar Integral (Rank 0)
id D1^-1*D2shift^-1 = B0(q.q,Melec,Melec);
.sort 

* Vector Tadpole
* The Symmetric Propagator (D1)
id k1(mu1?)*D1^-1 = 0;
* The Shifted Propagator (D2shift)
id k1(mu1?)*D2shift^-1 = q(mu1)*A0(Melec);
.sort 
* Leftover Tadpoles (Rank 0)
id D1^-1 = A0(Melec);
id D2shift^-1 = A0(Melec);
.sort

#message ">>>"
Print +s WardInt ;
Print +s BubbleInt ;
.sort;

Global BubbleToStore = BubbleInt;
.store
save QEDBubble.sav BubbleToStore;
.sort

.end

FORM 5.0.0 (Jan 27 2026, v5.0.0)                 Run: Mon Apr 20 18:01:24 2026
    
    * Process: ExperimentalLoop
    
    #-
    

   Bubble =

       + prop( - Melec^2 + k1.k1)*prop( - Melec^2 + k2.k2) * (
          + 4*q(mu1)*k1(mu2)*e^2
          + 4*q(mu2)*k1(mu1)*e^2
          - 8*k1(mu1)*k1(mu2)*e^2
          - 4*d_(mu1,mu2)*e^2*Melec^2
          - 4*d_(mu1,mu2)*q.k1*e^2
          + 4*d_(mu1,mu2)*k1.k1*e^2
          );

~~~">>>"

   Numerator =
       + 4*q(mu1)*k1(mu2)*e^2
       + 4*q(mu2)*k1(mu1)*e^2
       - 8*k1(mu1)*k1(mu2)*e^2
       - 4*d_(mu1,mu2)*e^2*Melec^2
       - 4*d_(mu1,mu2)*q.k1*e^2
       + 4*d_(mu1,mu2)*k1.k1*e^2
      ;

~~~">>>"

   Numerator =
       + 4*q(mu1)*k1(mu2)*e^2
       + 4*q(mu2)*k1(mu1)*e^2
       - 8*k1(mu1)*k1(mu2)*e^2
       + 2*d_(mu1,mu2)*e^2*D2shift
       + 2*d_(mu1,mu2)*e^2*D1
       - 2*d_(mu1,mu2)*q.q*e^2
      ;

~~~">>>"

   Numerator =
       + 4*q(mu1)*k1(mu2)*e^2
       + 4*q(mu2)*k1(mu1)*e^2
       - 8*k1(mu1)*k1(mu2)*e^2
     